# Submitting GPU jobs with Slurm

Interactive work is fine for developing. Real work goes through the **batch scheduler**: you describe what you need, hand it a script, and it runs when the resources are free.

This session runs a small Slurm-compatible scheduler. The commands, flags, output columns and job states are the ones you will use on Mahuika, so what you practise here transfers directly.

| Command | What it does |
|---|---|
| `sbatch script.sl` | Submit a job script to the queue |
| `squeue` | Show queued and running jobs |
| `squeue --me` | ...just yours |
| `scancel <jobid>` | Cancel a job |
| `sinfo` | Show the partitions and nodes |
| `sacct` | Show finished jobs too |
| `scontrol show job <jobid>` | Everything about one job |

> This is a teaching scaffold: one node, first-come-first-served, no fair-share or backfill. The job submission workflow is faithful; queueing policy is not.

## 1. What are we submitting to?

In [ ]:
!sinfo

## 2. Anatomy of a job script

A Slurm job script is an ordinary shell script with a block of `#SBATCH` comments at the top. Slurm reads those as command-line options; the shell ignores them as comments.

Let's write one that asks for **2 CPUs and 1 GPU**.

In [ ]:
%%writefile my_first_job.sl
#!/bin/bash -e
#SBATCH --job-name=first-gpu
#SBATCH --gpus-per-node=1
#SBATCH --cpus-per-task=2
#SBATCH --mem=2G
#SBATCH --time=00:05:00
#SBATCH --output=first-gpu-%j.out

echo "Job ${SLURM_JOB_ID} on ${SLURMD_NODENAME}"
echo "CPUs allocated : ${SLURM_CPUS_PER_TASK}"
echo "GPUs allocated : ${SLURM_GPUS_ON_NODE:-0}"
echo "GPU device IDs : ${SLURM_JOB_GPUS:-none}"
echo "CUDA_VISIBLE_DEVICES='${CUDA_VISIBLE_DEVICES}'"
echo

nvidia-smi

echo
echo "Doing some work on the GPU..."
gpuemu-burn --time 30 --memory 3GiB

Line by line:

- `--job-name` — what shows up in `squeue`. Make it recognisable; you will be looking for it among other people's jobs.
- `--gpus-per-node=1` — **the one that matters.** Without it you get no GPU. More on this below.
- `--cpus-per-task=2` — CPU cores. GPU jobs still need CPUs to feed the device.
- `--mem=2G` — *host* memory, not GPU memory. GPU memory is a property of the card you are given, not something you request.
- `--time=00:05:00` — wall-clock limit as `HH:MM:SS`. Your job is killed at this point, so ask for more than you need, but not wildly more: shorter jobs start sooner.
- `--output=first-gpu-%j.out` — where stdout and stderr go. `%j` is replaced with the job ID, which stops runs overwriting each other.

## 3. Submit it

In [ ]:
!sbatch my_first_job.sl

`sbatch` returns immediately with a job ID — it does **not** wait for the job to finish. Check on it with `squeue`:

In [ ]:
!squeue --me

The `ST` column is the job state. The ones you will actually meet:

| Code | State | Meaning |
|---|---|---|
| `PD` | PENDING | Waiting. `NODELIST(REASON)` says why. |
| `R` | RUNNING | Running now. |
| `CD` | COMPLETED | Finished, exit code 0. |
| `F` | FAILED | Finished, non-zero exit code. |
| `CA` | CANCELLED | You (or an admin) cancelled it. |
| `TO` | TIMEOUT | Hit its `--time` limit and was killed. |

While it runs, **open a terminal and run `nvtop`** to watch it. This is the habit worth forming: submit, then watch what the GPU actually does.

In [ ]:
# Run this a few times to watch the state change.
!squeue --me

## 4. Read the output

Once the job leaves the queue, its output is in the file named by `--output`.

In [ ]:
!ls -1 first-gpu-*.out && echo '---' && cat $(ls -t first-gpu-*.out | head -1)

`sacct` shows finished jobs, which `squeue` does not:

In [ ]:
!sacct

## 5. The mistake everyone makes

This is the most valuable cell in the notebook. Here is the same job **without** `--gpus-per-node`.

In [ ]:
%%writefile oops.sl
#!/bin/bash
#SBATCH --job-name=oops
#SBATCH --cpus-per-task=2
#SBATCH --mem=2G
#SBATCH --time=00:02:00
#SBATCH --output=oops-%j.out

echo "CUDA_VISIBLE_DEVICES='${CUDA_VISIBLE_DEVICES}'"
nvidia-smi || echo ">>> No GPU: none was requested."

In [ ]:
!sbatch oops.sl

In [ ]:
# Give it a moment to run, then:
!cat $(ls -t oops-*.out | head -1)

Notice what happened: the job **succeeded**. It did not crash, and nothing warned you. `CUDA_VISIBLE_DEVICES` was empty, so the GPU was invisible to it.

In real work this is insidious — PyTorch's usual

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

falls back to the CPU without complaint. Your training job runs, produces correct results, and takes fifty times longer than it should. People burn days of allocation this way.

**Defend against it**: put a check at the top of every GPU job so it fails loudly instead of quietly.

```bash
if [ -z "${CUDA_VISIBLE_DEVICES}" ]; then
    echo "ERROR: no GPU allocated - did you forget --gpus-per-node?" >&2
    exit 1
fi
```

or in Python:

```python
assert torch.cuda.is_available(), "expected a GPU, got none"
```

## 6. Cancelling

Submit something long, then stop it.

In [ ]:
!sbatch --job-name=longjob --gpus-per-node=1 --cpus-per-task=2 --time=01:00:00 \
        --wrap="gpuemu-burn --time 3600 --memory 2GiB"

In [ ]:
!squeue --me

In [ ]:
# Cancel every job of yours that is still active.
!scancel -u $USER
!squeue --me

`--wrap` above is worth remembering: it submits a single command without writing a script file. Handy for quick tests, but put anything you will run twice into a script.

## 7. A real training job

`~/gpu-training/examples/` has a complete PyTorch example. Submit it and watch with `nvtop`:

```bash
cd ~/gpu-training/examples
sbatch 03-train-pytorch.sl
squeue --me
```

## Exercises

1. Submit a job requesting **4 GPUs**. What does `sbatch` say, and why?
2. Write a job with `--time=00:00:30` that runs `gpuemu-burn --time 300`. What state does it end in, and where would you see that?
3. Submit two GPU jobs at once. Watch them with `squeue` — does the second start immediately? What does the `REASON` column say? (Depends on how many GPUs your session was started with.)
4. Add a `CUDA_VISIBLE_DEVICES` guard to `oops.sl` so it fails loudly. Confirm it now ends `FAILED` rather than `COMPLETED`.